# 🛠️ Notebook 2 · Online Stock Brokerage — Implementation

## 🛠️ Setup

```bash
cd 07-object-oriented-design/online-stock-brokerage
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


Here we turn the class design from Notebook 1 into **runnable code**: a tiny but functional brokerage.

We'll:
1. Build the domain classes.
2. Add order polymorphism (`MarketOrder`, `LimitOrder`, `StopOrder`).
3. Build a toy `Exchange` that matches orders when prices move.
4. Run end-to-end scenarios, including cancellation.


## 1️⃣ Domain: `Stock`, `Position`, `Portfolio`, `Account`

In [ ]:
from __future__ import annotations
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from datetime import datetime, timezone
from enum import Enum
from itertools import count

@dataclass
class Stock:
    symbol: str
    price: float      # current market price - kept simple, one number

class Side(Enum):
    BUY = "buy"
    SELL = "sell"

class OrderStatus(Enum):
    PENDING   = "pending"
    FILLED    = "filled"
    CANCELLED = "cancelled"

@dataclass
class Position:
    symbol: str
    qty: int = 0
    avg_price: float = 0.0   # weighted-average cost basis

@dataclass
class Portfolio:
    positions: dict[str, Position] = field(default_factory=dict)

    def apply(self, symbol: str, qty_delta: int, price: float) -> None:
        """Apply a buy (+qty) or sell (-qty). Updates avg_price on buys."""
        pos = self.positions.setdefault(symbol, Position(symbol))
        if qty_delta > 0:  # buying - update weighted avg cost
            new_qty = pos.qty + qty_delta
            pos.avg_price = (pos.avg_price * pos.qty + price * qty_delta) / new_qty
            pos.qty = new_qty
        else:              # selling - avg_price unchanged
            pos.qty += qty_delta
        if pos.qty == 0:
            self.positions.pop(symbol, None)

@dataclass
class Account:
    id: int
    cash: float
    portfolio: Portfolio = field(default_factory=Portfolio)


## 2️⃣ Orders — one class per strategy

In [ ]:
_oids = count(1)   # simple id generator for orders

class Order(ABC):
    def __init__(self, account: Account, stock: Stock, side: Side, qty: int):
        if qty <= 0:
            raise ValueError("qty must be positive")
        self.id = next(_oids)
        self.account = account
        self.stock = stock
        self.side = side
        self.qty = qty
        self.status = OrderStatus.PENDING

    @abstractmethod
    def can_fill(self, market_price: float) -> bool: ...
    @abstractmethod
    def fill_price(self, market_price: float) -> float: ...

    def __repr__(self):
        extra = ""
        if isinstance(self, LimitOrder): extra = f" limit={self.limit_price}"
        if isinstance(self, StopOrder):  extra = f" stop={self.stop_price}"
        return f"#{self.id} {type(self).__name__}({self.side.value} {self.qty} {self.stock.symbol}{extra}, {self.status.value})"


class MarketOrder(Order):
    def can_fill(self, market_price): return True
    def fill_price(self, market_price): return market_price

class LimitOrder(Order):
    def __init__(self, account, stock, side, qty, limit_price):
        super().__init__(account, stock, side, qty)
        self.limit_price = limit_price
    def can_fill(self, market_price):
        return (market_price <= self.limit_price) if self.side == Side.BUY \
               else (market_price >= self.limit_price)
    def fill_price(self, market_price): return market_price

class StopOrder(Order):
    def __init__(self, account, stock, side, qty, stop_price):
        super().__init__(account, stock, side, qty)
        self.stop_price = stop_price
    def can_fill(self, market_price):
        return (market_price >= self.stop_price) if self.side == Side.BUY \
               else (market_price <= self.stop_price)
    def fill_price(self, market_price): return market_price


## 3️⃣ The `Exchange` — match orders, record trades

A real exchange maintains an **order book** with price-time priority (we'll build that in Notebook 3). For now we use a simpler model: an order sits in the book until a `tick` moves the price into its fill zone.


In [ ]:
@dataclass
class Trade:
    order_id: int
    symbol: str
    side: Side
    qty: int
    price: float
    ts: datetime = field(default_factory=lambda: datetime.now(timezone.utc))

class Exchange:
    def __init__(self):
        self.book: list[Order] = []
        self.trades: list[Trade] = []

    # --- public API ------------------------------------------------------
    def place(self, order: Order) -> Order:
        """Accept an order. Try to fill immediately; otherwise rest in the book."""
        self.book.append(order)
        self._try_fill(order)
        return order

    def cancel(self, order_id: int) -> bool:
        """Cancel a resting (still pending) order by id."""
        for o in self.book:
            if o.id == order_id and o.status == OrderStatus.PENDING:
                o.status = OrderStatus.CANCELLED
                self.book.remove(o)
                return True
        return False

    def tick(self, symbol: str, new_price: float) -> None:
        """Market moved - try to fill any resting orders on that symbol."""
        for o in list(self.book):
            if o.stock.symbol == symbol and o.status == OrderStatus.PENDING:
                o.stock.price = new_price
                self._try_fill(o)

    # --- internal --------------------------------------------------------
    def _try_fill(self, o: Order) -> None:
        mp = o.stock.price
        if not o.can_fill(mp):
            return
        fp = o.fill_price(mp)
        cost = fp * o.qty

        if o.side == Side.BUY:
            if o.account.cash < cost:
                return  # not enough cash - stays pending
            o.account.cash -= cost
            o.account.portfolio.apply(o.stock.symbol, +o.qty, fp)
        else:  # SELL
            pos = o.account.portfolio.positions.get(o.stock.symbol)
            if not pos or pos.qty < o.qty:
                return  # no shorting in this toy - stays pending
            o.account.cash += cost
            o.account.portfolio.apply(o.stock.symbol, -o.qty, fp)

        o.status = OrderStatus.FILLED
        self.trades.append(Trade(o.id, o.stock.symbol, o.side, o.qty, fp))
        self.book.remove(o)


## 4️⃣ Scenario — Alice buys AAPL, then a limit fills on a dip

In [ ]:
ex = Exchange()
aapl = Stock("AAPL", 180.0)
alice = Account(id=1, cash=10_000)

# 1) Market buy: fills immediately at 180.
ex.place(MarketOrder(alice, aapl, Side.BUY, 10))

# 2) Limit buy below market: waits in the book.
lo = ex.place(LimitOrder(alice, aapl, Side.BUY, 5, limit_price=170))

print("After placing orders:")
print("  cash:", alice.cash, "| positions:", alice.portfolio.positions)
print("  book:", ex.book)

# 3) Price drops - the limit triggers.
ex.tick("AAPL", 168.0)

print("\nAfter price tick to 168:")
print("  cash:", alice.cash, "| positions:", alice.portfolio.positions)
print("  trades:")
for t in ex.trades: print("   ", t)


## 5️⃣ Scenario — cancel a resting order

In [ ]:
# Place a limit far from market, then cancel it before it can trigger.
far = ex.place(LimitOrder(alice, aapl, Side.BUY, 1, limit_price=50))
print("Before cancel, book:", ex.book)

ok = ex.cancel(far.id)
print("cancelled?", ok, "| status:", far.status, "| book:", ex.book)


## 6️⃣ Scenario — stop-loss protects a profit

Alice bought AAPL at 180 and it's now at 195. She places a **stop-sell at 190** so that if the price falls back through 190, she's automatically out with a small gain locked in.


In [ ]:
# Simulate the price rising to 195 (no resting orders on AAPL, so we update the stock directly).
aapl.price = 195.0

stop = ex.place(StopOrder(alice, aapl, Side.SELL, 10, stop_price=190))
print("Stop armed:", stop, "| book:", ex.book)

ex.tick("AAPL", 189.0)                     # price falls through the stop
print("After fall to 189:")
print("  cash:", alice.cash)
print("  positions:", alice.portfolio.positions)
print("  last trade:", ex.trades[-1])


## 7️⃣ Recap

- One class per order type → no `if/elif` on `kind` anywhere in `Exchange`.
- `Exchange.place`, `cancel`, and `tick` make a tiny but complete API.
- The order lifecycle (`PENDING → FILLED / CANCELLED`) lives on the `Order` itself.

### Try it yourself

- Add a `TrailingStopOrder` that moves its stop as the price rises. You should only need to **add one class** — existing code should keep working.
- Add `cash_reserved` to avoid double-spending when multiple buy orders are resting.
- Print P&L per position (`(current_price - avg_price) * qty`).

➡️ Notebook 3 adds a real **order book with price-time priority**, an **Observer** for live quotes, and a **Decorator** for risk checks.
